# Exhaustive Search Applications in Machine Learning

**Session:** 2026-10-28 · Brute-force Exhaustive  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

While exhaustive search is intractable for large state spaces, it remains the gold standard in machine learning when:
1. Exact global optimality is required for small combinatorial domains.
2. Generating ground-truth baselines to evaluate heuristics.
3. Conducting **Best Subset Selection** in feature selection.
4. Performing **Grid Search** hyperparameter tuning.

## Learning Outcomes

- Solve the 0/1 Knapsack Problem using brute-force subset enumeration.
- Implement Best Subset Selection to discover true sparse regression models.
- Build a Grid Search optimizer from scratch using Cartesian products.
- Execute unit test assertions to ensure end-to-end correctness.


## 1. 0/1 Knapsack Problem via Exhaustive Enumeration

Given $N$ items, each with weight $w_i$ and value $v_i$, and a backpack with capacity $W$:
$$\max \sum_{i \in S} v_i \quad \text{subject to} \quad \sum_{i \in S} w_i \le W$$

We enumerate all $2^N$ candidate subsets, discard infeasible ones, and choose the maximum utility.


In [ ]:
import csv
from itertools import combinations
from pathlib import Path


def solve_knapsack_brute_force(items: list[dict], capacity: int) -> tuple[int, int, list[str]]:
    """Exhaustively evaluate all 2^N item subsets to find global maximum."""
    n = len(items)
    best_value = 0
    best_weight = 0
    best_subset: list[str] = []

    for r in range(n + 1):
        for combo in combinations(items, r):
            tot_w = sum(it["weight"] for it in combo)
            tot_v = sum(it["value"] for it in combo)
            if tot_w <= capacity and tot_v > best_value:
                best_value = tot_v
                best_weight = tot_w
                best_subset = [it["name"] for it in combo]

    return best_value, best_weight, best_subset


# Load knapsack items dataset
knapsack_path = Path("../data/knapsack_items.csv")
catalog = []
with knapsack_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        catalog.append(
            {
                "item_id": int(row["item_id"]),
                "name": row["name"],
                "weight": int(row["weight"]),
                "value": int(row["value"]),
            }
        )

cap = 8
max_val, used_w, chosen = solve_knapsack_brute_force(catalog, capacity=cap)
print(f"Knapsack capacity {cap}: Maximum Value = {max_val}, Weight Used = {used_w}/{cap}")
print("Selected Items:", chosen)
assert used_w <= cap
assert max_val >= 25

## 2. Best Subset Selection in Linear Regression

In linear regression with $p$ features, **Best Subset Selection** tests all $2^p - 1$ non-empty feature combinations. It solves ordinary least squares for each subset and selects the subset minimizing the residual sum of squares (RSS) or BIC.


In [ ]:
import csv
from pathlib import Path

import numpy as np


def fit_linear_regression(x_matrix: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, float]:
    """Compute OLS weights (X^T X)^{-1} X^T y and residual sum of squares."""
    # Add intercept column
    x_aug = np.c_[np.ones(x_matrix.shape[0]), x_matrix]
    weights = np.linalg.pinv(x_aug.T @ x_aug) @ x_aug.T @ y
    preds = x_aug @ weights
    rss = float(np.sum((y - preds) ** 2))
    return weights, rss


def best_subset_selection(x_features: np.ndarray, y_target: np.ndarray, feature_names: list[str]):
    """Exhaustively evaluate all 2^p - 1 feature subsets."""
    p = x_features.shape[1]
    best_score = float("inf")
    best_features = []
    best_weights = None

    for k in range(1, p + 1):
        for combo in combinations(range(p), k):
            x_sub = x_features[:, combo]
            weights, rss = fit_linear_regression(x_sub, y_target)
            # BIC penalty = n * ln(RSS / n) + k * ln(n)
            n_samples = len(y_target)
            bic = n_samples * np.log(rss / n_samples) + (k + 1) * np.log(n_samples)

            if bic < best_score:
                best_score = bic
                best_features = [feature_names[i] for i in combo]
                best_weights = weights

    return best_features, best_weights, best_score


# Load feature selection dataset
data_path = Path("../data/feature_selection_data.csv")
with data_path.open("r", encoding="utf-8") as f:
    reader = csv.reader(f)
    headers = next(reader)
    rows = [[float(v) for v in row] for row in reader]

arr = np.array(rows)
feat_names = headers[:5]
x_data = arr[:, :5]
y_data = arr[:, 5]

selected_feats, weights, score = best_subset_selection(x_data, y_data, feat_names)
print(f"Best subset selected features: {selected_feats}")
print(f"Model BIC: {score:.3f}")
# True model used features x1, x2, x4 (x3 and x5 were pure noise)
assert "x1" in selected_feats
assert "x2" in selected_feats
assert "x4" in selected_feats
print("Best Subset Selection recovered the true sparse generative model!")

## 3. Grid Search Hyperparameter Tuning

In machine learning, finding optimal hyperparameters (e.g. learning rate, regularization, tree depth) is formulated as an exhaustive search over the Cartesian product of discrete candidate sets:
$$\Theta^* = \arg\min_{\theta \in \Theta_1 \times \dots \times \Theta_k} \mathcal{L}_{\text{val}}(f_\theta)$$


In [ ]:
from collections.abc import Callable
from itertools import product
from typing import Any


def grid_search(
    param_grid: dict[str, list[Any]], objective_fn: Callable[[dict[str, Any]], float]
) -> tuple[dict[str, Any], float]:
    """Exhaustively test all Cartesian combinations of hyperparameters."""
    keys = list(param_grid.keys())
    values_product = list(product(*param_grid.values()))

    best_loss = float("inf")
    best_params: dict[str, Any] = {}

    for combo in values_product:
        params = dict(zip(keys, combo, strict=True))
        loss = objective_fn(params)
        if loss < best_loss:
            best_loss = loss
            best_params = params

    return best_params, best_loss


# Benchmark simulated objective function
def loss_surface(params: dict[str, Any]) -> float:
    lr = params["learning_rate"]
    reg = params["l2_reg"]
    # convex quadratic minimum at lr=0.01, reg=0.001
    return (lr - 0.01) ** 2 * 1000 + (reg - 0.001) ** 2 * 100000 + 0.15


grid = {"learning_rate": [0.1, 0.05, 0.01, 0.001], "l2_reg": [0.01, 0.005, 0.001, 0.0001]}

best_p, min_l = grid_search(grid, loss_surface)
print("Optimal hyperparameters found:", best_p)
print(f"Minimum validation loss: {min_l:.4f}")
assert best_p["learning_rate"] == 0.01
assert best_p["l2_reg"] == 0.001

## Summary and Curriculum Context

- **Brute force & exhaustive search** provide fundamental guarantees: completeness and global optimality.
- In algorithm design, they establish the theoretical benchmark against which dynamic programming, branch-and-bound, greedy algorithms, and gradient-based methods are compared.
- In practical machine learning, exhaustive search powers exact nearest-neighbor baselines, feature subset selection, and hyperparameter grid search.
